In [3]:
import pandas as pd
import numpy as np
from datetime import datetime

# Load datasets
employees_df = pd.read_csv("datasets/Employees_Dataset_Final_With_Shifts.csv")
vacation_df = pd.read_csv("datasets/vacation_requests_updated.csv")
attendance_df = pd.read_csv("datasets/attendance_logs_updated.csv")

# Convert date columns to datetime objects
vacation_df['start_date'] = pd.to_datetime(vacation_df['start_date'])
vacation_df['end_date'] = pd.to_datetime(vacation_df['end_date'])
attendance_df['date'] = pd.to_datetime(attendance_df['date'])

KeyError: 'date'

In [ ]:
# Calculate Total Vacation Days Taken per Employee
# Assuming 'status' == 'Approved'
vacation_df['days_requested'] = (vacation_df['end_date'] - vacation_df['start_date']).dt.days + 1
vacation_metrics = vacation_df[vacation_df['status'] == 'Approved'].groupby('Employee_ID')['days_requested'].sum().reset_index()
vacation_metrics.columns = ['Employee_ID', 'Total_Vacation_Days_Taken']

# Calculate Attendance Metrics
# Assuming columns: 'Employee_ID', 'Hours_Worked', 'Is_Late'
attendance_metrics = attendance_df.groupby('Employee_ID').agg({
    'Hours_Worked': ['sum', 'mean'],
    'Is_Late': 'sum'
}).reset_index()

# Flatten multi-index columns
attendance_metrics.columns = ['Employee_ID', 'Total_Hours_Worked', 'Avg_Daily_Hours', 'Late_Count']

Note: you may need to restart the kernel to use updated packages.


ERROR: unknown command "i"



In [ ]:
# Identify columns to drop from the original employee dataset (Adjust names based on your file)
cols_to_drop = ['Old_Total_Hours', 'Old_Vacation_Used', 'Static_Attendance_Score'] 
employees_df = employees_df.drop(columns=[c for c in cols_to_drop if c in employees_df.columns])

# Merge datasets
final_df = employees_df.merge(attendance_metrics, on='Employee_ID', how='left')
final_df = final_df.merge(vacation_metrics, on='Employee_ID', how='left').fillna(0)

Feature Engineering for Attrition RiskWe calculate risk indicators using data mining principles. High overtime combined with low vacation usage often signals burnout.Burnout Index: Ratio of overtime to vacation taken.Engagement Score: Frequency of late arrivals vs. tenure.The Attrition Risk Score ($R$) can be modeled as:$$R = w_1 \left( \frac{H_{total}}{H_{avg} \cdot D} \right) + w_2 (L) - w_3 (V_{taken})$$Where:$H$ = Hours$L$ = Late count$V$ = Vacation days$w$ = Weight coefficients

In [ ]:
# Standardize metrics for scoring (0 to 1 scale)
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

# Feature: Burnout (High hours, Low vacation)
final_df['Overtime_Factor'] = normalize(final_df['Total_Hours_Worked'])
final_df['Vacation_Neglect'] = 1 - normalize(final_df['Total_Vacation_Days_Taken'])

# Feature: Reliability (Late arrivals)
final_df['Late_Factor'] = normalize(final_df['Late_Count'])

# Calculate Attrition Risk Score (Weighted Average)
final_df['Attrition_Risk_Score'] = (
    (final_df['Overtime_Factor'] * 0.4) + 
    (final_df['Vacation_Neglect'] * 0.3) + 
    (final_df['Late_Factor'] * 0.3)
) * 100

# Categorize Risk
final_df['Risk_Level'] = pd.cut(final_df['Attrition_Risk_Score'], 
                                bins=[0, 30, 70, 100], 
                                labels=['Low', 'Medium', 'High'])

print(final_df[['Employee_ID', 'Attrition_Risk_Score', 'Risk_Level']].head())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Correlation analysis to find drivers of risk
correlation_matrix = final_df[['Total_Hours_Worked', 'Late_Count', 'Total_Vacation_Days_Taken', 'Attrition_Risk_Score']].corr()

plt.figure(figsize=(10, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='RdYlGn')
plt.title("Drivers of Attrition Risk")
plt.show()